In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score, GridSearchCV

In [3]:
df = pd.read_csv("diabetes.csv")

In [4]:
print("features of Dataset", df.shape)

features of Dataset (768, 9)


In [5]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [7]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [8]:
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [12]:
print(df.columns.tolist())

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


In [15]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [17]:
svm = SVC()
svm.fit(X_train, y_train)
pred = svm.predict(X_test)
acc_before = accuracy_score(y_test, pred)
f1_before = f1_score(y_test, pred)
print("Accuracy and F1 score before SVM :", round(acc_before, 4), " ", round(f1_before, 4))

Accuracy and F1 score before SVM : 0.7338   0.6019


In [18]:
scores = cross_val_score(SVC(), X_train, y_train, cv=5)

print("Scores of each fold :", scores.round(4))
print("Mean CV accuracy :", round(scores.mean(), 4))

Scores of each fold : [0.7724 0.8049 0.7317 0.7561 0.7787]
Mean CV accuracy : 0.7687


In [19]:
param_grid = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": [0.001, 0.01, 0.1, 1],
}

grid = GridSearchCV(SVC(), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best parameters :", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 4))

Best parameters : {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
Best CV accuracy: 0.772


In [20]:
best_svm = grid.best_estimator_
pred_best = best_svm.predict(X_test)

acc_after = accuracy_score(y_test, pred_best)
f1_after = f1_score(y_test, pred_best)

print("After tuning -> Accuracy:", round(acc_after, 4), " F1:", round(f1_after, 4))

After tuning -> Accuracy: 0.7727  F1: 0.6789


In [21]:
result = pd.DataFrame({
    "": ["Before (default)", "After (Grid Search)"],
    "CV accuracy": [scores.mean(), grid.best_score_],
    "Test accuracy": [acc_before, acc_after],
    "Test F1": [f1_before, f1_after],
})
print(result.round(4))

                        CV accuracy  Test accuracy  Test F1
0     Before (default)       0.7687         0.7338   0.6019
1  After (Grid Search)       0.7720         0.7727   0.6789
